# 06 - Full Pipeline (End-to-End)

Complete workflow from data generation through training to evaluation, all in one notebook.

Ideal for quick prototyping and verifying the entire pipeline works end-to-end.

**Tip**: Set `JAX_PLATFORM_NAME=cpu` before launching Jupyter for fast compilation during debugging.

In [ ]:
import sys
import jax
import numpy as np
import matplotlib.pyplot as plt

from dpjax.config import load_config, merge_config
from dpjax.paths import PROJECT_ROOT, DATA_DIR, RUNS_DIR, ensure_dir

print(f"JAX backend: {jax.default_backend()}")
print(f"JAX devices: {jax.devices()}")
print(f"Project root: {PROJECT_ROOT}")

## Step 1: Generate Data

In [ ]:
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
from plummer.plummer_gendata import sample_df, save_data

N = 131072
eta = sample_df(N, max_dist=10.0)

ensure_dir(DATA_DIR)
DATA_PATH = DATA_DIR / "plummer_n131072.h5"
save_data(eta, str(DATA_PATH))
print(f"Generated {eta.shape[0]} samples -> {DATA_PATH}")

## Step 2: Train DF (RealNVP)

In [ ]:
from experiments.train_df import run_df_training

df_cfg = load_config("configs/df_plummer.yaml")
df_cfg = merge_config(df_cfg, {
    "train": {"epochs": 4, "log_every": 20, "ckpt_every": 100}
})

DF_RUN = RUNS_DIR / "plummer" / "df"
df_result = run_df_training(df_cfg, DATA_PATH, DF_RUN)
print(f"DF trained, final step: {df_result['final_step']}")

## Step 3: Train Phi (Potential)

In [ ]:
from experiments.train_phi import run_phi_training

phi_cfg = load_config("configs/phi_plummer.yaml")
phi_cfg = merge_config(phi_cfg, {
    "train": {"epochs": 4, "log_every": 20, "ckpt_every": 100}
})

PHI_RUN = RUNS_DIR / "plummer" / "phi"
phi_result = run_phi_training(phi_cfg, DATA_PATH, DF_RUN, PHI_RUN)
print(f"Phi trained, final step: {phi_result['final_step']}")

## Step 4: Evaluate

In [ ]:
from experiments.eval_phi import run_eval_phi

eval_res = run_eval_phi(DATA_PATH, DF_RUN, PHI_RUN, n_eval=8192)

print("Residual statistics:")
for k, v in eval_res["stats"].items():
    print(f"  {k}: {v}")

In [ ]:
rad = eval_res["radial"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

ax1.plot(rad["r"], rad["phi_true"], "k-", lw=2, label="Plummer analytic")
ax1.plot(rad["r"], rad["phi_learned_shift"], "--", lw=1.5, label="Learned")
ax1.set_xscale("log"); ax1.set_xlabel("r"); ax1.set_ylabel(r"$\Phi(r)$")
ax1.legend(); ax1.set_title("Gravitational Potential")
ax1.grid(True, alpha=0.2)

ax2.plot(rad["r"], rad["ar_true"], "k-", lw=2, label="Plummer analytic")
ax2.plot(rad["r"], rad["ar_learned"], "--", lw=1.5, label="Learned")
ax2.set_xscale("log"); ax2.set_xlabel("r"); ax2.set_ylabel(r"$a_r(r)$")
ax2.legend(); ax2.set_title("Radial Acceleration")
ax2.grid(True, alpha=0.2)

fig.suptitle("Plummer Sphere: Learned vs Analytic", fontsize=13)
fig.tight_layout()
plt.show()

## Step 5: (Optional) Joint Fine-Tuning

In [ ]:
from experiments.finetune_joint import run_joint_finetuning

joint_cfg = load_config("configs/joint_plummer.yaml")
joint_cfg = merge_config(joint_cfg, {
    "train": {
        "epochs": 2,
        "log_every": 20,
        "ckpt_every": 100,
        "mode": "alt",
    }
})

JOINT_RUN = RUNS_DIR / "plummer" / "joint"
joint_result = run_joint_finetuning(
    joint_cfg, DATA_PATH, DF_RUN, PHI_RUN, JOINT_RUN,
)
print(f"Joint fine-tuning complete, final step: {joint_result['final_step']}")

In [ ]:
# Compare before/after joint fine-tuning
joint_eval = run_eval_phi(DATA_PATH, JOINT_RUN / "df", JOINT_RUN / "phi", n_eval=8192)

print("\n=== Before joint fine-tuning ===")
for k, v in eval_res["stats"].items():
    print(f"  {k}: {v}")

print("\n=== After joint fine-tuning ===")
for k, v in joint_eval["stats"].items():
    print(f"  {k}: {v}")